# Prueba Intertrimestral

**Nombre: Gabriela**  
**Apellidos: De Dorremochea Blanco**  

**Tiempo de la prueba:** 2 horas  
**Asignatura:** Desarrollo de Aplicaciones para la Visualización de Datos  
**Fecha:** 19 de octubre de 2027  

## Instrucciones

- Escribe **código limpio y autoexplicativo**.  
- Se pueden utilizar **los materiales de clase**.  
- Se puede utilizar **internet** para búsqueda de dudas y documentación.  
- **No se puede utilizar ningún tipo de LLM** (ChatGPT, Copilot, Gemini, etc.).  
- **No se puede utilizar mensajería instantánea**.  
- Al finalizar, **sube tu notebook a GitHub** y envía el enlace del fichero según las instrucciones del profesor.

Al finalizar:

**[Subé el enlace aquí](https://forms.cloud.microsoft/e/vSwHis0f1B)**


## Dataset “City Micromobility Hourly”

El fichero `city_micromobility_hourly.csv` contiene **alquileres horarios** de bicicletas y patinetes compartidos en una ciudad (ene 2025 – dic 2026): meteorología, tipo de día, usuarios ocasionales/abonados, duración media, ingresos y una etiqueta de alta demanda.

Hay un `.txt` con la descripción de las variables en la misma carpeta.


## Inicialización de librerías


In [177]:
# Librerías que vamos a usar en todo el examen
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, confusion_matrix

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


# Ejercicio 1 — Programación (2 puntos)

a) (0.6 pts) Crea una función `rental_fee(duration_min, bike_type, is_member)` que calcule la tarifa de un viaje en euros, redondeada a **dos decimales**, con estas reglas:

- Tarifa base: **1.00 €** si `bike_type == "classic"`, **1.50 €** si `bike_type == "electric"`.
- Coste por minuto: **0.05 €/min** (classic) o **0.08 €/min** (electric).
- Si `is_member` es `True`, aplica un **20 % de descuento** sobre el total.
- Si `bike_type` no es `"classic"` ni `"electric"`, lanza `ValueError`.

Ejemplo orientativo: `rental_fee(20, "classic", False)` → `2.00`.


In [55]:
def rental_fee(duration_min, bike_type, is_member):
    "Calcula la tarifa de un viaje en euros (redondeada a 2 decimales)."

    # Tarifa según el tipo de bici
    if bike_type == "classic":
        base = 1.00
        por_minuto = 0.05
    elif bike_type == "electric":
        base = 1.50
        por_minuto = 0.08
    else:
        raise ValueError("bike_type debe ser 'classic' o 'electric'")

    # Total sin descuento
    total = base + por_minuto * duration_min

    # Con descuento (20 %) si es abonado
    if is_member == True:
        total = total * 0.8

    return round(total, 2)


# ALgunas pruebas para la comprobacion
print("20 min, classic, no abonado ->", rental_fee(20, "classic", False))   
print("20 min, classic, abonado    ->", rental_fee(20, "classic", True))    
print("10 min, electric, no abonado ->", rental_fee(10, "electric", False))
print("10 min, electric, abonado    ->", rental_fee(10, "electric", True))  

# Lanza el error con un tipo raro para la comprobacion
try:
    rental_fee(10, "scooter", False)
except ValueError as error:
    print("Error capturado:", error)

20 min, classic, no abonado -> 2.0
20 min, classic, abonado    -> 1.6
10 min, electric, no abonado -> 2.3
10 min, electric, abonado    -> 1.84
Error capturado: bike_type debe ser 'classic' o 'electric'


b) (0.6 pts) Crea una función `demand_label(total_rentals, threshold)` que devuelva `"Alta"` si `total_rentals >= threshold`, `"Media"` si está entre `threshold/2` (incluido) y `threshold` (excluido), y `"Baja"` en caso contrario.


In [56]:
def demand_label(total_rentals, threshold):
    "Devuelve 'Alta', 'Media' o 'Baja' según el número de alquileres."
    if total_rentals >= threshold:
        return "Alta"
    elif total_rentals >= threshold / 2:
        return "Media"
    else:
        return "Baja"



print(demand_label(30,70))

Baja


In [31]:

# Pruebas con threshold = 40
print("40 ->", demand_label(40, 40))   
print("20 ->", demand_label(20, 40))  
print("19 ->", demand_label(19, 40))  

40 -> Alta
20 -> Media
19 -> Baja


c) (0.6 pts) Genera un DataFrame llamado `tripsDemo` con **al menos 10 filas** y columnas:

- `duration_min` (float)
- `bike_type` (`"classic"` o `"electric"`)
- `is_member` (bool)
- `fee` (calculada con `rental_fee`)
- `total_rentals` (int, inventado)
- `demand` (calculada con `demand_label`, usando por ejemplo `threshold=40`)

Muestra el DataFrame completo.


In [44]:
# Datos random
duraciones = [5.0, 12.5, 20.0, 8.0, 30.0, 15.0, 45.0, 3.5, 25.0, 10.0, 18.0, 60.0]
tipos      = ["classic", "electric", "classic", "electric", "classic", "electric",
              "electric", "classic", "classic", "electric", "classic", "electric"]
abonados   = [True, False, False, True, True, False, True, False, True, False, False, True]
alquileres = [55, 12, 38, 70, 20, 5, 41, 19, 80, 33, 9, 60]

tripsDemo = pd.DataFrame({
    "duration_min": duraciones,
    "bike_type": tipos,
    "is_member": abonados,
    "total_rentals": alquileres,
})


# Tarifa fila a fila 
tarifas = []
for i in range(len(tripsDemo)):
    fila = tripsDemo.iloc[i]
    tarifas.append(rental_fee(fila["duration_min"], fila["bike_type"], fila["is_member"]))
tripsDemo["fee"] = tarifas


# Etiqueta de demanda con threshold = 30
etiquetas = []
for valor in tripsDemo["total_rentals"]:
    etiquetas.append(demand_label(valor, 30))
tripsDemo["demand"] = etiquetas

# Orden de las columnas 
tripsDemo = tripsDemo[["duration_min", "bike_type", "is_member", "fee", "total_rentals", "demand"]]


In [48]:
print(tripsDemo)

    duration_min bike_type  is_member   fee  total_rentals demand
0            5.0   classic       True  1.00             55   Alta
1           12.5  electric      False  2.50             12   Baja
2           20.0   classic      False  2.00             38   Alta
3            8.0  electric       True  1.71             70   Alta
4           30.0   classic       True  2.00             20  Media
5           15.0  electric      False  2.70              5   Baja
6           45.0  electric       True  4.08             41   Alta
7            3.5   classic      False  1.18             19  Media
8           25.0   classic       True  1.80             80   Alta
9           10.0  electric      False  2.30             33   Alta
10          18.0   classic      False  1.90              9   Baja
11          60.0  electric       True  5.04             60   Alta


In [49]:
print(tripsDemo.dtypes)

duration_min     float64
bike_type            str
is_member           bool
fee              float64
total_rentals      int64
demand               str
dtype: object


d) (0.2 pts) Integra ambas funciones en una clase `MicromobilityTools` que, a partir de listas/arrays de `duration_min`, `bike_type` e `is_member`, calcule las tarifas y (opcionalmente) etiquete demanda a partir de una lista de `total_rentals` y un `threshold`.


In [36]:
class MicromobilityTools:

_IncompleteInputError: incomplete input (1104974939.py, line 1)

## Ejercicio 2 — Exploración y visualización (3 puntos)

a) (0.75 pts) Carga `city_micromobility_hourly.csv` en un DataFrame `mobilityData`. Muestra las 5 primeras filas y el número de filas y columnas.


In [111]:
mobilityData = pd.read_csv("city_micromobility_hourly.csv")


In [112]:
#Primeras 5 filas
print(mobilityData.head())

              datetime  year  month  day  hour  weekday  season  is_holiday  \
0  2025-01-01 00:00:00  2025      1    1     0        2  Winter           1   
1  2025-01-01 01:00:00  2025      1    1     1        2  Winter           1   
2  2025-01-01 02:00:00  2025      1    1     2        2  Winter           1   
3  2025-01-01 03:00:00  2025      1    1     3        2  Winter           1   
4  2025-01-01 04:00:00  2025      1    1     4        2  Winter           1   

   is_weekend  is_workingday  ... wind_kmh  precip_mm  casual_users  \
0           0              0  ...      9.7        0.0             2   
1           0              0  ...      2.8        0.0             1   
2           0              0  ...      9.0        0.0             1   
3           0              0  ...      5.4        0.0             2   
4           0              0  ...      2.4        0.0             1   

   member_users  total_rentals  avg_trip_min  avg_distance_km  electric_share  \
0             8  

In [113]:
print("Número de filas:", mobilityData.shape[0])
print("Número de columnas:", mobilityData.shape[1])

Número de filas: 17520
Número de columnas: 23


b) (0.75 pts) Crea una función `describeData(dataFrame)` que, para cada columna, devuelva: tipo de dato, número de nulos y porcentaje de nulos.


In [114]:
# Creacion de funcion
def describeData(dataFrame):
    "Devuelve, para cada columna, el tipo de dato, el nº de nulos y el % de nulos."
    resumen = pd.DataFrame()
    resumen["tipo"] = dataFrame.dtypes
    resumen["nulos"] = dataFrame.isnull().sum()
    resumen["pct_nulos"] = round(dataFrame.isnull().sum() / len(dataFrame) * 100, 2)
    return resumen

c) (0.75 pts) Usa `describeData(mobilityData)` y comenta brevemente (2–3 líneas) qué variables tienen más valores faltantes y qué harías con ellas antes de modelar.


In [115]:
resumen = describeData(mobilityData)
print(resumen)
print()

# Solo  columnas con algún nulo
con_nulos = resumen[resumen["nulos"] > 0].sort_values("nulos", ascending=False)
print("Columnas con nulos:")
print(con_nulos)
print()
print("Filas con al menos un nulo:", mobilityData.isnull().any(axis=1).sum(), "de", len(mobilityData))

                    tipo  nulos  pct_nulos
datetime             str      0       0.00
year               int64      0       0.00
month              int64      0       0.00
day                int64      0       0.00
hour               int64      0       0.00
weekday            int64      0       0.00
season               str      0       0.00
is_holiday         int64      0       0.00
is_weekend         int64      0       0.00
is_workingday      int64      0       0.00
weather              str      0       0.00
temp_c           float64      0       0.00
humidity         float64    224       1.28
wind_kmh         float64    142       0.81
precip_mm        float64      0       0.00
casual_users       int64      0       0.00
member_users       int64      0       0.00
total_rentals      int64      0       0.00
avg_trip_min     float64    255       1.46
avg_distance_km  float64      0       0.00
electric_share   float64    182       1.04
revenue_eur      float64      0       0.00
high_demand

Las variables con mas valores nulos son: humidity, wind_kmh, avg_trip_min y electric_share. Hay varias opciones para hacer con ellas antes de modelar una de ellas seria rellenarlas con la media del resto que si que hay o tambien podriamos desechar esos elementos. 

d) (0.75 pts) Elabora **dos** visualizaciones:

1. Un gráfico de correlación entre variables numéricas relevantes (por ejemplo `temp_c`, `humidity`, `wind_kmh`, `precip_mm`, `casual_users`, `member_users`, `total_rentals`, `revenue_eur`).
2. Un gráfico que muestre la demanda media (`total_rentals`) por `hour` (y, si quieres, separando fin de semana / laborable).

Comenta 2–3 hallazgos (correlaciones, horas punta, efecto del clima, etc.).


In [116]:
# Gráfico de correlacion entre variables numericas relevantes
columnas_corr = ["temp_c", "humidity", "wind_kmh", "precip_mm",
                 "casual_users", "member_users", "total_rentals", "revenue_eur"]

# Matriz de correlacion 
matriz_corr = mobilityData[columnas_corr].corr()
print(matriz_corr.round(2))

fig = px.imshow(matriz_corr.round(2), text_auto=True, color_continuous_scale="RdBu_r",
                zmin=-1, zmax=1, title="Correlación entre variables numéricas")
fig.show()

               temp_c  humidity  wind_kmh  precip_mm  casual_users  \
temp_c           1.00     -0.22      0.01      -0.11          0.38   
humidity        -0.22      1.00     -0.01       0.01         -0.15   
wind_kmh         0.01     -0.01      1.00      -0.00         -0.03   
precip_mm       -0.11      0.01     -0.00       1.00         -0.17   
casual_users     0.38     -0.15     -0.03      -0.17          1.00   
member_users     0.40     -0.15     -0.05      -0.16          0.57   
total_rentals    0.44     -0.17     -0.05      -0.18          0.82   
revenue_eur      0.41     -0.16     -0.04      -0.18          0.94   

               member_users  total_rentals  revenue_eur  
temp_c                 0.40           0.44         0.41  
humidity              -0.15          -0.17        -0.16  
wind_kmh              -0.05          -0.05        -0.04  
precip_mm             -0.16          -0.18        -0.18  
casual_users           0.57           0.82         0.94  
member_users         

In [117]:
# Demanda media por hora, separando laborable / no laborable
demanda_hora = mobilityData.groupby(["hour", "is_workingday"])["total_rentals"].mean().reset_index()
demanda_hora["tipo_dia"] = demanda_hora["is_workingday"].map({1: "Laborable", 0: "Fin de semana / festivo"})
print(demanda_hora.head(10))

fig = px.line(demanda_hora, x="hour", y="total_rentals", color="tipo_dia", markers=True,
              title="Demanda media (total_rentals) por hora del día",
              labels={"hour": "Hora", "total_rentals": "Alquileres medios", "tipo_dia": "Tipo de día"})
fig.show()


   hour  is_workingday  total_rentals                 tipo_dia
0     0              0      12.127273  Fin de semana / festivo
1     0              1       9.380392                Laborable
2     1              0      11.881818  Fin de semana / festivo
3     1              1       9.168627                Laborable
4     2              0      11.472727  Fin de semana / festivo
5     2              1       9.231373                Laborable
6     3              0      11.800000  Fin de semana / festivo
7     3              1       9.341176                Laborable
8     4              0      11.318182  Fin de semana / festivo
9     4              1       9.211765                Laborable


Correlaciones: 
"total_rentals" tiene una correlación muy alta con "member_users" (0,94), "casual_users" (0,82) y "revenue_eur" (0,91). Todas estas variables son partes de los alquileres, así que no se pueden usar para predecir. 
Además las variables de clima, la temperatura es la que más se relaciona con la demanda (0,44), la lluvia (−0,18) y la humedad (−0,17) la reducen un poco.


Horas punta:
En días laborables hay dos picos, de 7h a 9h y de 17h a 19h (70 alquileres/h el doble que el resto del día). Se puede relacionar con la ida y la vuelta del trabajo. En fin de semana no hay picos. La demanda es casi plana y máxima al mediodía.

Clima:
Con cielo despejado hay de media 40 alquileres/h, con lluvia solo 16.

## Ejercicio 3 — Regresión (2 puntos)

Queremos predecir `total_rentals`.

a) (0.25 pts) Define `targetVar = "total_rentals"` y selecciona **3–5** variables predictoras numéricas razonables (p. ej. `hour`, `temp_c`, `humidity`, `wind_kmh`, `precip_mm`, `is_workingday`, `is_weekend`). Construye `X` e `y`, tratando nulos de forma simple (p. ej. eliminar filas o imputar la mediana).


In [135]:
targetVar = "total_rentals"

# NO uso casual_users, member_users ni revenue_eur (por lo de antes de que están correlacionadas)
predictoras = ["hour", "temp_c", "humidity", "precip_mm", "is_workingday"]

X = mobilityData[predictoras].copy()
y = mobilityData[targetVar]

print("Nulos en X antes de imputar:")
print(X.isnull().sum())

# La media en las columnas con nulos
X = X.fillna(X.mean())


Nulos en X antes de imputar:
hour               0
temp_c             0
humidity         224
precip_mm          0
is_workingday      0
dtype: int64


In [136]:
print("Nulos en X")
print(X.isnull().sum())

Nulos en X
hour             0
temp_c           0
humidity         0
precip_mm        0
is_workingday    0
dtype: int64


b) (0.25 pts) Divide en entrenamiento (70 %) y prueba (30 %) con `random_state=42`.


In [137]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [138]:
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

X_train: (12264, 5)
X_test:  (5256, 5)
y_train: (12264,)
y_test:  (5256,)


c) (0.75 pts) Entrena un `LinearRegression`, calcula **R²** y **MSE** en el conjunto de prueba y comenta el resultado en 1–2 líneas.


In [139]:
modelo_lineal = LinearRegression()
modelo_lineal.fit(X_train, y_train)

pred_lineal = modelo_lineal.predict(X_test)

r2_lineal = r2_score(y_test, pred_lineal)
mse_lineal = mean_squared_error(y_test, pred_lineal)

print("Regresión lineal")
print("R²  =", round(r2_lineal, 4))
print("MSE =", round(mse_lineal, 2))
print()
print("Coeficientes:")
for nombre, coef in zip(predictoras, modelo_lineal.coef_):
    print(f"  {nombre}: {coef:.3f}")

Regresión lineal
R²  = 0.3206
MSE = 408.66

Coeficientes:
  hour: 1.047
  temp_c: 0.973
  humidity: -0.131
  precip_mm: -9.428
  is_workingday: -1.847


La regresión lineal solo explica el 32% de la variabilidad. Es bajo porque la relación con la hora no es una recta si no que la demanda sube y baja dos veces al día, una por la mañana y otra por la tarde.

d) (0.75 pts) Entrena un `RandomForestRegressor` (mismo split) y compara R² y MSE con la regresión lineal. ¿Cuál funciona mejor y por qué crees que es así?


In [140]:
modelo_rf = RandomForestRegressor(n_estimators=100, random_state=42)
modelo_rf.fit(X_train, y_train)

pred_rf = modelo_rf.predict(X_test)

r2_rf = r2_score(y_test, pred_rf)       
mse_rf = mean_squared_error(y_test, pred_rf)

print("Random Forest")
print("R²  =", round(r2_rf, 4))
print("MSE =", round(mse_rf, 2))
print()

comparacion = pd.DataFrame({
    "modelo": ["Regresión lineal", "Random Forest"],
    "R2": [round(r2_lineal, 4), round(r2_rf, 4)],
    "MSE": [round(mse_lineal, 2), round(mse_rf, 2)],
})
print(comparacion)

Random Forest
R²  = 0.883
MSE = 70.36

             modelo      R2     MSE
0  Regresión lineal  0.3206  408.66
1     Random Forest  0.8830   70.36


El Random Forest funciona mejor. Esto es porque tiene un mayor R² = 0,88 y un MSE casi mucho menor. Esto pasa porque los árboles pueden aprender relaciones no lineales y combinaciones entre variables.

## Ejercicio 4 — Clasificación (2 puntos)

Usa la columna `high_demand` (1 = demanda alta, 0 = no).

a) (0.25 pts) Comprueba el balance de clases (`value_counts` o proporción) de `high_demand`.


In [167]:
print("Recuento de clases")
print(mobilityData["high_demand"].value_counts())

Recuento de clases
high_demand
0    12071
1     5449
Name: count, dtype: int64


In [168]:
print("Proporción de clases")
print(mobilityData["high_demand"].value_counts(normalize=True).round(3))

Proporción de clases
high_demand
0    0.689
1    0.311
Name: proportion, dtype: float64


In [169]:
# Si siempre fuese 0
baseline = mobilityData["high_demand"].value_counts(normalize=True).max()
print()
print("Accuracy si siempre fuese 0 :", round(baseline, 3))


Accuracy si siempre fuese 0 : 0.689


b) (0.75 pts) Entrena un `LogisticRegression` para predecir `high_demand` usando variables de contexto (hora, clima, tipo de día, etc.). Recuerda preparar `X`/`y` y el split train/test.


In [170]:
# Como la demanda no es lineal, para que la logística lo pueda captar, creamos una columna de 'hora punta'
mobilityData["hora_punta"] = 0
mobilityData.loc[mobilityData["hour"].isin([7, 8, 9, 17, 18, 19]), "hora_punta"] = 1

# Y otra de la noche
mobilityData["noche"] = 0
mobilityData.loc[mobilityData["hour"].isin([23, 0, 1, 2, 3, 4, 5]), "noche"] = 1

In [171]:
print(mobilityData[["hour", "hora_punta", "noche"]].drop_duplicates().sort_values("hour"))

    hour  hora_punta  noche
0      0           0      1
1      1           0      1
2      2           0      1
3      3           0      1
4      4           0      1
5      5           0      1
6      6           0      0
7      7           1      0
8      8           1      0
9      9           1      0
10    10           0      0
11    11           0      0
12    12           0      0
13    13           0      0
14    14           0      0
15    15           0      0
16    16           0      0
17    17           1      0
18    18           1      0
19    19           1      0
20    20           0      0
21    21           0      0
22    22           0      0
23    23           0      1


In [172]:
variables_clf = ["hora_punta", "noche", "temp_c", "humidity", "precip_mm", "is_workingday"]

X_clf = mobilityData[variables_clf].copy()
X_clf = X_clf.fillna(X_clf.median())
y_clf = mobilityData["high_demand"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=42, stratify=y_clf)

print("Train:", X_train_c.shape, " Test:", X_test_c.shape)

# Escalamos para que los coeficientes sean comparables entre sí
escalador = StandardScaler()
X_train_c_esc = escalador.fit_transform(X_train_c)
X_test_c_esc = escalador.transform(X_test_c)

modelo_log = LogisticRegression(max_iter=1000)
modelo_log.fit(X_train_c_esc, y_train_c)

Train: (12264, 6)  Test: (5256, 6)


,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

c) (0.25 pts) Calcula la **accuracy** y muestra la **matriz de confusión**.


In [173]:
pred_log = modelo_log.predict(X_test_c_esc)

accuracy = accuracy_score(y_test_c, pred_log)
print("Accuracy del modelo:", round(accuracy, 4))


Accuracy del modelo: 0.8449


In [174]:
matriz = confusion_matrix(y_test_c, pred_log)
print("Matriz de confusión")
print(matriz)

fig = px.imshow(matriz, text_auto=True, color_continuous_scale="Blues",
                x=["Predicho 0", "Predicho 1"], y=["Real 0", "Real 1"],
                title="Matriz de confusión — regresión logística")
fig.show()

Matriz de confusión
[[3251  370]
 [ 445 1190]]


d) (0.75 pts) Interpreta qué factores parecen asociarse más a la alta demanda (coeficientes del modelo logístico y/o importancia si usas un árbol; o argumentación basada en tus gráficos del ejercicio 2).


In [175]:
coeficientes = pd.DataFrame({"variable": variables_clf, "coeficiente": modelo_log.coef_[0]})
coeficientes = coeficientes.sort_values("coeficiente")
print(coeficientes)

fig = px.bar(coeficientes, x="coeficiente", y="variable", orientation="h",
             title="Coeficientes del modelo logístico")
fig.show()

        variable  coeficiente
1          noche    -3.099575
4      precip_mm    -1.421417
5  is_workingday    -0.394039
3       humidity    -0.389055
2         temp_c     1.115435
0     hora_punta     1.206983


El modelo acierta un 84,5 % así que aporta información verdadera. En la matriz de confusión se ve que falla más con las horas de alta demanda que con las normales.

Las variables están escaladas y el tamaño del coeficiente indica cuánto pesa cada una:
 "noche" tiene el factor más fuerte: de noche casi nunca hay demanda alta.
 "precip_mm": la lluvia reduce la probabilidad de demanda.
 "hora_punta" y "temp_c": las horas punta y el calor la aumentan.
 
Estos resultados son los mismos que en los gráficos del ejercicio 2, la hora del día y el tiempo son lo que más mueve la demanda.

## Ejercicio 5 — Conclusión (1 punto)

En un máximo de **300 palabras**, resume:

* Qué factores (hora, clima, tipo de día, etc.) parecen mover más la demanda de micromovilidad.  
* Qué modelo predictivo te ha funcionado mejor y por qué.  
* Qué mejorarías con más tiempo o datos (más ciudades, eventos, precio dinámico, etc.).


Al finalizar:

**[Subé el enlace aquí](https://forms.cloud.microsoft/e/vSwHis0f1B)**



Factores que más mueven la demanda.

 El factor más importante para la demanda es la hora del día. Ya que de noche (23–5 h) no hay casi alquileres y pasa lo contrario en días laborables (por los desplazamientos al trabajo). El segundo factor es el clima. Pasa que cuando hay más temperatura también hay más alquileres y la lluvia los reduce a menos de la mitad. 


Mejor modelo.

Para predecir el número de alquileres, el Random Forest (R² = 0,88) supera a la regresión lineal (R² = 0,32). Para la clasificación de alta demanda, la regresión logística alcanza un 84,5 % de accuracy, esto pasa por crear variables sencillas (hora punta y noche). 


Mejoras con más tiempo o datos. 

Usaría la hora como variable categórica o un modelo de arboles para clasificar. Tmabién probaría validación cruzada y separaría train/test por fecha para comprobar que los resultados son fiables. Si tuviese más datos añadiría datos de otras ciudades o precios que pudiesen variar.